In [ ]:
# 获取token
import requests
import os

login_url = "https://admin.summerfarm.net/authentication/auth/username/login"
login_data = {
    "username": "peng.tang@summerfarm.net",
    "password": os.getenv("XIANMU_ADMIN_PASSWORD"),
}

token = requests.post(login_url, data=login_data).json()

print(token)


headers = {
    "token": token.get("data").get("token"),
    "xm-rqid": "create_fake_merchant_tp",
    "xm-uid": "2047",
    "Content-Type": "application/json;charset=UTF-8",
}

print(headers)

In [ ]:
large_area_name_to_create_sub_area = "武汉大区"


def get_large_area_by_name(name=large_area_name_to_create_sub_area):
    url = "https://admin.summerfarm.net/large-area/1/200"
    response = requests.get(url, headers=headers).json()
    for area in response.get("data").get("list"):
        if name in area.get("largeAreaName"):
            return area
    return None


area = get_large_area_by_name()
print(area)

In [ ]:
import random

charactors_to_use = "QAZWSXEDCRFVTGBYHNUJMIKOPL"


def generate_random_area_name(length=3):
    random_chars = "".join(random.choice(charactors_to_use) for _ in range(length))
    return "测试A" + random_chars


# Example usage:
print(generate_random_area_name())
print(generate_random_area_name(6))

In [ ]:
## 创建虚拟的运营区域（用来给新建的大区添加空的小区）

import requests

url = "https://admin.summerfarm.net/area/add"
data = {
    "adminId": 2047,
    "freeDay": "",
    "level": 2,
    "areaName": f"{generate_random_area_name()}",
    "companyAccountId": 5,
    "companyName": "杭州鲜沐科技有限公司",
    "administrativeArea": f"新疆维吾尔自治区/{large_area_name_to_create_sub_area}测试",
    "status": 0,
    "deliveryFee": 0,
    "expressFee": 0,
    "mapSection": [],
    "deliveryRule": '{"categoryList":[]}',
    "memberRule": "[]",
    "weChatNotify": 0,
    "notifyTitle": "",
    "notifyContent": "",
    "notifyRemarks": "",
    "nextDeliveryDate": "",
    "originAreaNo": "",
    "closeOrderType": "",
    "allocationNextTimeType": "",
    "nextDayReach": "",
    "payChannel": "",
    "largeAreaName": f"{large_area_name_to_create_sub_area}测试小区",
    "largeAreaNo": area.get("largeAreaNo"),
    "supportAddOrder": 1,
    "warehouseList": [],
}
print("data:", data)
response = requests.put(url, headers=headers, json=data)

print(response.status_code)
print(response.text)

## 日常http 请求

In [ ]:
import urllib.parse
import requests
import numpy as np
import time

server_rt = []
server_rt_mall = []
query_list = """黄油,奶油,蛋糕,淡奶油,巧克力,奶酪,安佳黄油,糖,芝士,可可粉,安德鲁,芝士片,越南大青芒,果酱,慕斯,糖粉,果糖,糖浆,芝士碎,
低粉,提拉米苏,动物奶油,芒果酱,魔客,鲜奶,稀奶油,维益,红颜,大成,椰,酸奶,咖啡奶,西米,高筋粉,咖奶,奶,玉米淀粉,肠,椰奶,嘉利宝,
饼干,低筋,冷冻蛋糕,芝士粉,巧克力酱,红颜草莓（盆装）,澄善,杏仁片,乳酪,杏仁粉""".split(
    ","
)
for query in query_list:
    query = urllib.parse.quote(query)
    url = f"https://qah5.summerfarm.net/product/1/60?pdName={query}"
    token = "mall__c92a4ec3-d038-4420-b126-6296b55675fc"
    start_time = time.time()
    response = requests.get(url, headers={"token": token})
    end_time = time.time()
    server_rt.append(int((end_time - start_time) * 1000))

    start_time_mall = time.time()
    response_mall = requests.get(url.replace("qah5", "h5"), headers={"token": token})
    end_time_mall = time.time()
    server_rt_mall.append(int((end_time_mall - start_time_mall) * 1000))
    time.sleep(0.5)

print(
    f"server_rt quantile info: {np.quantile(server_rt, [0, 0.25, 0.5, 0.75, .90,.95,1])}",
    f"server_rt_mall quantile info: {np.quantile(server_rt_mall, [0, 0.25, 0.5, 0.75, .90,.95,1])}",
)

In [ ]:
np.quantile(server_rt, [0, 0.25, 0.5, 0.75, 0.90, 0.95, 1])

In [ ]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta

api_path = """/topic/page/query/sku
/recommend/1/6
/home/query/commonly-recommended
/banner/query/expired
/product/1/6
/product-coupon/1/6
/temporary/fix/1/6"""

api_path = api_path.split("\n")
query_template = """ap:__api_path__|select json_extract(ai, '$.qh') as query_header,uid,json_extract_scalar(ai, '$.data') data_send,ap,
json_extract(ai, '$.params') params,
json_extract_scalar(ai, '$.method')method, json_extract_scalar(ai, '$.rt') data_returned order by __time__ desc limit 50"""

from_time = datetime.now() - timedelta(hours=4)
end_time = datetime.now()
project = "xianmu-front-end-log"
logstore = "xm-mall"

df_list = []
for api in api_path:
    query = query_template.replace("__api_path__", api)

    _df = get_sls_data_by_query(
        query=query,
        from_time=from_time,
        to_time=end_time,
        project=project,
        logstore=logstore,
    )
    df_list.append(_df)

In [ ]:
df_list[0]

In [ ]:
import json

import requests
from urllib.parse import urlencode


def send_data(
    path, method: str = "POST", jsonObject=None, params: dict = {}, headers={}
) -> bool:
    """Send data to the server."""
    host_list = ["https://test619.summerfarm.net", "https://h5.summerfarm.net"]
    data_1, data_2 = {}, {}
    trace1, trace2 = "", ""
    for index, host in enumerate(host_list):
        url = host + path

        if params:
            url = f"{url}?{urlencode(params)}"
            print(f"params:{params}, url:{url}")

        try:
            response = requests.request(
                method=method.upper(),
                url=url,
                json=jsonObject,
                headers=headers,
            )
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            if index == 0:
                data_1 = response.json().get("data", {})
                trace1 = response.headers.get("eagleeye-traceid")
            else:
                data_2 = response.json().get("data", {})
                trace2 = response.headers.get("eagleeye-traceid")
        except requests.exceptions.RequestException as e:
            print(
                f"请求报错！！！{e},{response.text if 'response' in locals() else ''}"
            )
            return False
    if data_1 != data_2:
        print(
            "数据不一致!!!\n\n",
            path,
            f"\n{trace1}",
            data_1,
            f"\n{trace2}",
            data_2,
        )
        return False
    return True


for df in df_list:
    for index, row in df.iterrows():
        if index>2:
            break
        query_header = json.loads(row["query_header"])

        headers_to_send = {
            "Accept": "application/json, text/plain, */*",
            "Content-Type": query_header.get("Content-Type", "application/json"),
            "token": query_header.get("token", ""),
            "xm-ab-exp": query_header.get("xm-ab-exp", ""),
        }

        uid = row["uid"]
        data_send = row["data_send"]
        if data_send and data_send != "null":
            data_send = json.loads(data_send)
        else:
            data_send = None
        params = row["params"]
        try:
            if params and params != "null":
                params = json.loads(params)
            else:
                params = None
        except Exception as e:
            print(e, f"\n出错的ROW:{row.to_dict()}")
        ap = row["ap"]
        method = row["method"]
        data_returned = row["data_returned"]
        result = send_data(
            path=ap,
            method=method,
            jsonObject=data_send,
            params=params,
            headers=headers_to_send,
        )
        if result:
            # print(f"数据不一致, uid:{uid}, data_send:{data_send}, ap:{ap}\n\n")
        # else:
            print(f"数据一致, uid:{uid}, ap:{ap}")